# Experimento — KAN, versão mais recente

Placeholder do experimento. O texto abaixo é o roteiro. Não há treino neste arquivo.

Rede de Kolmogorov–Arnold treinada do zero. No lugar de uma ativação fixa em cada neurônio, cada aresta carrega uma função univariada aprendida. A linha de referência deste placeholder é a KAN 2.0, pacote `pykan` 0.2.8, a última versão publicada no PyPI quando o roteiro foi escrito, já com os nós de multiplicação do MultKAN. A conferência desta pesquisa não achou release posterior a 0.2.8 no PyPI. Se sair uma da mesma linha antes do bloco 0, o experimento usa essa.


## Como o experimento avança

O experimento é uma sequência de blocos. Cada bloco faz uma intervenção só: um hiperparâmetro, uma escolha de arquitetura ou uma mudança nos dados. O bloco seguinte parte da configuração que venceu, e só fica no lugar dela se a métrica de validação subir.

A métrica é escolhida uma vez, antes do bloco 0 de qualquer modelo, e vale para os sete notebooks. Cerca de 26,5% dos clientes cancelam, então a métrica tem de enxergar a classe de churn. O conjunto de teste não entra nessa escolha. Ele só é usado no bloco final, com a configuração já congelada.

Empate fica com a configuração mais simples. Cada bloco aceito registra o que mudou, a configuração completa e a média e a dispersão nas cinco dobras.

O gradient boosting deste repositório é a referência de comparação. Cada modelo anota a distância do seu melhor bloco até o boosting congelado com o mesmo protocolo. O boosting também anda por blocos; a comparação usa a versão já aceita, sem perseguir um número que ainda está mudando.


## Dados

A entrada é a saída de `if702-preprocessamento.ipynb`: as cinco dobras estratificadas e a função `preparar` para o treino final e o teste. Em cada dobra o scaler é ajustado só no treino da dobra.

O oversampling repetitivo já está feito aí. O `RandomOverSampler` só duplica o treino da dobra. A validação e o teste continuam com um cliente por linha. Nenhum bloco refaz esse balanceamento por fora das dobras.

### Estratégias de dados compartilhadas

Nenhuma, além do pré-processamento.

Quando aparecer uma estratégia nova de exploração ou de mineração, ela é escrita nesta seção e copiada para os outros seis notebooks. Vira um bloco em cada experimento, em cima da configuração que aquele modelo já tiver aceito. O bloco mexe só nos dados. O modelo fica parado.


## Frentes já definidas

A versão oficial no PyPI continua a 0.2.8, de 14 de novembro de 2024. Não há release posterior. O treino nessa versão é `fit`. O README ainda escreve `model.train`; esse método não existe no código da tag. `KAN` nessa tag é `MultKAN`.

`fit` com a perda vazia usa erro quadrático. Para o churn, a perda de classificação é escolhida uma vez, antes do bloco 0. Não é um knob da busca.

Cada argumento abaixo é um bloco.

- `width`. Lista com a entrada, as camadas ocultas e a saída. Um inteiro vira só nós de soma. O construtor não tem default de largura. Sem faixa oficial. O README manda encolher `grid` antes de `width` quando treino e teste se afastam.
- `grid`. Número de intervalos da grade do spline. Default 3. Sem faixa oficial.
- `k`. Ordem do B-spline. Default 3. Sem faixa oficial.
- `base_fun`. Default `'silu'`. O construtor só converte `'silu'`, `'identity'` e `'zero'`. Logística, tangente hiperbólica e ReLU não são valores desse argumento. Tanh aparece na biblioteca simbólica, que fica fora desta busca.
- `grid_eps`. Default 0,02, no intervalo documentado de 0 a 1. 1 é grade uniforme e 0 é a partição pelos percentis da amostra.
- `opt`. Default `"LBFGS"`. O outro valor documentado é `"Adam"`. Não há SGD nem RMSprop nesse `fit`.
- `lr`. Default 1,0. Sem faixa oficial. Fica separado do otimizador: a fonte não diz que os dois interagem.
- `steps`. Default 100. Sem faixa oficial.
- `batch`. Default −1, o lote inteiro. Com cerca de 7 mil linhas, o default já é lote cheio.
- `update_grid`. Default `True`. Reposiciona a grade que já existe e não aumenta o número de intervalos. Entram no mesmo bloco os companheiros que só existem para agendar isso: `grid_update_num` (default 10), `start_grid_update_step` (default −1) e `stop_grid_update_step` (default 50).
- `refine(new_grid)`. É um método, não um argumento de `fit`. Sobe o número de intervalos em cima de um modelo já aceito. O exemplo da docstring vai de 5 para 10. O README avisa overfitting.
- Família `lamb`. Com `lamb` em 0, que é o default, o resto da penalidade não entra. `lamb_l1` default 1, `lamb_entropy` default 2, `lamb_coef` default 0 e `lamb_coefdiff` default 0. O README manda começar com `lamb=0`. Não há dropout. A regularização própria da KAN é esta família.
- `scale_base_mu` (default 0) e `scale_base_sigma` (default 1) desenham a magnitude residual inicial. `sb_trainable` e `sp_trainable`, os dois default `True`, decidem se essas magnitudes entram no otimizador. Não existe argumento `scale_spline` no construtor de `MultKAN`.
- Multiplicação. No `width`, o segundo número do par é a quantidade de nós de multiplicação da camada. `mult_arity` default 2, o número de entradas de cada um desses nós. Essa frente só abre depois da KAN só com somas.

Fica de fora o ramo simbólico: `fix_symbolic`, `auto_symbolic`, `prune` e a leitura de fórmula. O README descreve esse ramo para descoberta científica, não para subir a métrica de uma classificação.

O oversampling repetitivo das dobras vale no `fit`.

Fontes: pykan 0.2.8 no PyPI, `kan/MultKAN.py` na tag v0.2.8, e os tutoriais de API 4, 5, 6 e 8 em kindxiaoming.github.io/pykan.


## Hiperparâmetros que ainda vão chegar

A pesquisa destes hiperparâmetros está nas frentes acima. O que ainda chegar entra como bloco novo, um item de cada vez, a partir da configuração vencedora.


## Ordem prevista dos blocos

1. Bloco 0. KAN só com somas, `grid` 3, `k` 3, `base_fun` `'silu'`, `opt` `"LBFGS"`, `lr` 1, `steps` 100, lote cheio, `lamb` 0, `update_grid` ligado. A perda já é a de classificação.
2. `width`. Largura, depois profundidade.
3. `grid`.
4. `k`.
5. `opt`, `"Adam"` contra o LBFGS já aceito.
6. `lr`.
7. `steps`.
8. `batch`, se o lote cheio não for o vencedor do bloco 0.
9. `base_fun`: `'identity'` e `'zero'`, contra `'silu'`.
10. `grid_eps`.
11. `update_grid` e o agendamento dele, num bloco só.
12. `refine`, só em cima da configuração já aceita.
13. `lamb`. Se a penalidade vencer, um relativo por vez: `lamb_entropy`, `lamb_l1`, `lamb_coef`, `lamb_coefdiff`.
14. `scale_base_mu` e `scale_base_sigma`. Congelar `sb_trainable` ou `sp_trainable` é um bloco seguinte, se for aberto.
15. Nós de multiplicação no `width`, e só então `mult_arity`.
16. Estratégias de dados compartilhadas.
17. Bloco final no teste.


## O que este arquivo ainda não contém

Não há código, treino, busca nem número medido. Este notebook é o roteiro. O primeiro bloco só começa depois que a métrica comum estiver escrita.
